**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Foundations of Signal Processing (2)

The sequel [Part 1](./Foundations_of_Signal_Processing_1.ipynb) promised: the z-transform as the discrete world's native language, multirate processing (changing sample rates without lying), the polyphase trick that makes it cheap, and a first meeting with wavelets.

## 1. Pre-requisites

- [Part 1](./Foundations_of_Signal_Processing_1.ipynb) Sessions 3–8.
- [Complex Analysis Lite](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb) — poles, ROC, residues (used throughout Session 1).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *The z-Transform & ROC* (~35 min)
**Goal:** master the discrete transform: ROC geometry, stability, and inversion by partial fractions.
**Builds on:** [Part 1](./Foundations_of_Signal_Processing_1.ipynb) S4–S5; [Complex Analysis](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb). &nbsp; **Feeds into:** Session 2 (multirate).

---

## 2. The z-Transform

💡 **Intuition.** The z-transform $X(z) = \sum_n x[n] z^{-n}$ is the DTFT with a volume knob: on $z = re^{j\omega}$, it's the DTFT of $x[n] r^{-n}$ — signals too wild for Fourier become tame after exponential damping. The **ROC** records which damping levels work, and it carries real information: the *same* algebraic $X(z)$ with different ROCs describes different signals (causal vs anticausal). Stability = ROC contains the unit circle; causality = ROC extends outward.

**The table you can now derive** (via residues, [Complex Analysis S2](../Intro_Math/Complex_Analysis/Complex_Analysis_Lite.ipynb)):

| $x[n]$ | $X(z)$ | ROC |
|---|---|---|
| $\delta[n]$ | $1$ | all $z$ |
| $a^n u[n]$ | $\frac{z}{z-a}$ | $|z| > |a|$ |
| $-a^n u[-n-1]$ | $\frac{z}{z-a}$ | $|z| < |a|$ ← same formula, different signal! |
| $r^n \sin(\theta n) u[n]$ | ratio with poles $re^{\pm j\theta}$ | $|z| > r$ |

**Key properties:** delay $x[n-k] \leftrightarrow z^{-k}X(z)$ (why filters are polynomials in $z^{-1}$), convolution ↔ multiplication.

In [ ]:
# The ROC is not decoration: one X(z), two signals — only the ROC disambiguates

# YOUR CODE HERE


**What just happened.** Two completely different signals, **one algebraic expression**. Both panels are $X(z) = z/(z - 1.25)$. The left is causal — zero for $n < 0$, then growing without bound. The right is anticausal — zero for $n \geq 0$, decaying backwards into the past, and perfectly bounded.

Nothing distinguishes them except the ROC. $|z| > 1.25$ selects the causal one; $|z| < 1.25$ selects the anticausal one. **So the ROC is not a technical footnote — it is half of the answer.** A z-transform quoted without its ROC is genuinely ambiguous, and writing down $z/(z-a)$ and stopping specifies nothing.

**Two rules turn that into something usable.** Stability means the ROC contains the unit circle; causality means it extends outward from the outermost pole. Combine them for a pole at $1.25$, outside the circle: causality forces $|z| > 1.25$, which excludes the unit circle, so the system cannot be stable. Stability forces an ROC containing $|z| = 1$, hence $|z| < 1.25$, so it cannot be causal.

**With a pole outside the unit circle, stability and causality are mutually exclusive — you choose one.** That is a structural constraint, not a limitation of any particular design method.

**And it is a choice you have already made in practice.** Choosing stability over causality means running a filter that needs future samples — exactly `filtfilt` in [Filter Design](./Filter_Design.ipynb), whose backward pass is deliberately anticausal and legal only because the signal is already recorded. A bedside monitor cannot make that choice; an offline pipeline can.

Worth noting what makes any of this possible: on the unit circle the z-transform *is* the DTFT, and off it, $z = re^{j\omega}$ gives the DTFT of $x[n]r^{-n}$. The radius is a damping knob, and the ROC records which damping levels make the sum converge.

---
### 🕐 Session 2 of 4 — *Multirate: Decimation & Interpolation* (~40 min)
**Goal:** change sample rates honestly: anti-alias before dropping, filter after stuffing.
**Builds on:** Session 1; [Part 1](./Foundations_of_Signal_Processing_1.ipynb) S5 (sampling). &nbsp; **Feeds into:** Session 3 (polyphase).

---

## 3. Changing the Sample Rate

💡 **Intuition.** **Downsampling** by $M$ (keep every $M$-th sample) stretches the spectrum by $M$ — anything beyond the new Nyquist folds back as aliasing, so you must low-pass *first* (decimation = filter + downsample). **Upsampling** by $L$ (insert $L-1$ zeros) compresses the spectrum and reveals $L-1$ spectral *images* — ghosts of the original — which the interpolation filter must erase. Every resampler, DAC, and neural 'stride/transposed conv' is these two moves.

In [ ]:

# YOUR CODE HERE


**What just happened.** Same signal, two ways of reducing its rate by 4. Naive `x[::4]` leaves a large spurious peak that was never in the signal; `sig.decimate` — which low-passes *first* — leaves only the 40 Hz tone we wanted.

**Work the arithmetic, because it is easy to get almost right.** After decimating by 4 the rate is 250 Hz, so the new Nyquist is 125 Hz. The 380 Hz intruder is $380/250 = 1.52$ cycles per new sample; the fractional part is 0.52, which exceeds a half, so it folds to $1 - 0.52 = 0.48$ cycles/sample — that is $0.48 \times 250 = \mathbf{120}$ Hz.

Note the trap: $|380 - 250| = 130$ is a tempting answer and it is wrong, because 130 exceeds the 125 Hz Nyquist and must itself fold. *(The plot title originally read 130 Hz and has been corrected.)* Getting this right requires reducing modulo the sample rate **and then** folding about Nyquist — two steps, and skipping the second is the standard slip.

**And the aliased peak is indistinguishable from a real signal.** A genuine 120 Hz tone would produce exactly the same samples. No downstream processing can separate them, because they are not merely similar — after sampling they are the *same data*. That is why aliasing is worse than noise: noise degrades a measurement, aliasing replaces it with a plausible wrong one.

**Which fixes the ordering rule.** Decimation is **filter, then downsample**. Filtering afterwards is too late: the fold has already happened and the two components are already identical. Ask what a filter applied to `naive` could possibly do — it can attenuate 120 Hz, but that removes a frequency the real signal might also occupy, and it cannot recover what was destroyed.

This is the [Part 1](./Foundations_of_Signal_Processing_1.ipynb) sampling theorem restated as a procedure, and it is the same rule as the anti-alias filter in front of an ADC, the optical blur in front of a camera sensor ([Image Processing](./Image_Processing.ipynb)), and the decimation filter after a [sigma-delta modulator](./Sigma_Delta_Quantization.ipynb). One theorem, four hardware consequences.

In [ ]:
# Upsampling: zero-stuffing creates images; the interpolation filter erases them

# YOUR CODE HERE


**What just happened.** Zero-stuffing by 4 produces **three spectral images** — copies of the 40 Hz tone at higher frequencies that were not audible before. After proper interpolation filtering, only the original remains.

**Where the images come from, precisely.** Inserting zeros adds no information whatsoever; the underlying samples are unchanged. What changes is the *sample rate*, and therefore what the frequency axis means. Content that previously sat above the old Nyquist is now inside the new band and becomes visible. The spectrum did not move — the axis rescaled around it. Students find this genuinely counterintuitive, and the useful framing is that upsampling is a *relabelling* that exposes structure which was always implicitly there.

**Which gives the mirror-image rule.** Interpolation is **upsample, then filter** — the opposite ordering to decimation, and for a symmetric reason. In decimation the damage (aliasing) happens *during* the rate change, so the filter must come first. In interpolation the artifacts (images) are *created* by the rate change, so the filter must come after. Ask the room to state both rules and explain why they differ; getting that symmetry is worth more than memorising two recipes.

**Note what `resample_poly` is doing.** It upsamples and filters in one operation, and — as Session 3 shows — it never computes the zeros it would be multiplying by. That is the polyphase optimisation, already at work in a library call the room has been using without noticing.

**And this escapes DSP entirely.** A strided convolution in a neural network is decimation; a transposed convolution is upsampling by zero-stuffing followed by a *learned* filter. The notorious checkerboard artifacts in GAN-generated images are interpolation images that the learned filter failed to suppress — the exact phenomenon in the left panel, appearing in a paper about image generation. Naming that connection lands well with an ML-inclined room and shows these rules are not parochial.

---
### 🕐 Session 3 of 4 — *Polyphase Structures* (~35 min)
**Goal:** never compute what you'll throw away: the decomposition behind every efficient resampler.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (wavelets).

---

## 4. The Polyphase Trick

💡 **Intuition.** Filter-then-downsample wastes $\frac{M-1}{M}$ of its work computing outputs that get discarded. The polyphase fix: split the filter into $M$ interleaved sub-filters (its *phases*), run each at the **low** rate on the input's interleaved streams, and sum. Identical output, $M\times$ cheaper — pure bookkeeping, no approximation. This structure is why sample-rate conversion is cheap enough to be everywhere, and it's the skeleton of the filter banks in Session 4.

In [ ]:
# Polyphase decimation by hand — verify exact equivalence, then time it
# reference: full-rate filter, then discard 3 of every 4 outputs
# scipy's polyphase engine computes ONLY the surviving outputs

# YOUR CODE HERE


**What just happened.** Two numbers, and they should be read very differently.

**The exactness is the strong claim.** `max |polyphase − reference| = 1.6e-15` — machine precision. Polyphase decimation is not a fast *approximation* of filter-then-downsample; it is the identical computation, reorganised. Every output bit-for-bit the same, with the arithmetic that would have been discarded simply never performed. That is worth stating plainly, because "faster" methods in signal processing usually trade accuracy, and this one does not.

The idea is pure bookkeeping. Filtering at full rate and keeping every $M$-th output wastes $(M-1)/M$ of the work — 75% at $M = 4$. Polyphase splits the filter into $M$ interleaved sub-filters, splits the input the same way, runs each at the **low** rate, and sums. Same answer, none of the waste.

**The speedup, though, needs care: theory says 4×, the stopwatch says 1.7×.** That gap is real and worth understanding rather than ignoring. Several things cause it. `np.convolve` on $2^{18}$ samples is already highly optimised C and may internally switch to FFT-based convolution, so the "slow" baseline is not naive. `upfirdn` carries Python-level call overhead that a 2 ms measurement cannot amortise. And at this data size the computation is substantially **memory-bandwidth bound** rather than arithmetic bound, so removing multiplications does not remove the bottleneck.

The general lesson is one this curriculum returns to in [Performance Engineering](../Intro_GPU/Performance_Engineering.ipynb): **operation counts model arithmetic, and real machines are usually limited by something else.** An algorithmic 4× becomes a wall-clock 4× only when arithmetic is genuinely the constraint — which is exactly the situation on an embedded DSP or in FPGA fabric, where there is no optimised BLAS underneath and every multiplier is silicon you paid for. That is where polyphase earns its full factor, and it is why the structure is universal in hardware resamplers even though a laptop benchmark understates it.

Note also that these timings are machine-dependent and taken from a single run; treat 1.7× as "meaningfully faster, less than theory," not as a measurement to quote.

---
### 🕐 Session 4 of 4 — *Wavelets, a First Meeting* (~40 min)
**Goal:** trade the STFT's fixed window for scale: the Haar transform, coded from scratch.
**Builds on:** Session 3; [Part 1](./Foundations_of_Signal_Processing_1.ipynb) S6 (uncertainty).

---

## 5. Beyond Fixed Windows

💡 **Intuition.** The STFT slices time with ONE window length — so it resolves either the click or the pitch well, never both ([Part 1 S6](./Foundations_of_Signal_Processing_1.ipynb)'s uncertainty trade, frozen in). Wavelets spend the uncertainty budget *adaptively*: short windows for high frequencies, long for low — constant-Q tiling. Implementation-wise a wavelet transform is just a two-channel filter bank (low-pass + high-pass, downsample by 2) applied **recursively to the low-pass branch** — Session 3's machinery, iterated.

In [ ]:
# The Haar wavelet transform from scratch: averages & differences, recursively
# a piecewise-constant "blocks" signal — Fourier's nightmare, wavelets' lunch

# YOUR CODE HERE


**What just happened.** `perfect reconstruction: True` — the forward transform followed by the inverse returns the original signal exactly. Nothing was lost.

That is worth pausing on, because it is easy to assume a wavelet transform *is* compression. It is not. The Haar transform is an orthonormal change of basis, exactly like the DFT: same information, different coordinates, fully invertible. Compression happens when you *discard coefficients afterwards*, which is the next cell. Keeping the two ideas separate matters — the transform is lossless, the thresholding is where the loss lives and where the design choices are.

**Look at what the code actually does, because it is smaller than the theory suggests.** `(approx[0::2] + approx[1::2])/√2` is an average of sample pairs — a two-tap **low-pass followed by ↓2**. The difference is the matching **high-pass followed by ↓2**. Then the loop recurses *on the low-pass branch only*.

So a wavelet transform is a two-channel filter bank applied recursively — Sessions 2 and 3's machinery, iterated. The $\sqrt2$ normalisations are what make it orthonormal, so energy is preserved and Parseval holds exactly as in the DFT. There is no new theory here; there is a familiar structure arranged in a tree.

**And that recursion is what produces constant-Q tiling.** Each level halves the rate, so each successive level analyses a band an octave lower with twice the time support. High frequencies get short windows (good timing, coarse frequency resolution); low frequencies get long ones (the reverse). Compare with the STFT, which fixes one window length and therefore one rectangle shape everywhere.

Crucially, this does **not** beat the uncertainty principle from [Part 1 S6](./Foundations_of_Signal_Processing_1.ipynb) — every tile has the same area. Wavelets spend the same budget differently, allocating resolution where the signal is likely to need it. The bet is that real signals have brief high-frequency events and sustained low-frequency content, which is true often enough to be useful — and it is roughly how the ear's critical bands are arranged too.

In [ ]:
# Compression bake-off at equal budget: keep the 40 largest coefficients
# wavelet: threshold across all detail levels
# Fourier: keep top-k magnitude bins (hermitian pairs counted once)

# YOUR CODE HERE


**What just happened.** Equal budget — 40 coefficients each — and Haar reconstructs the piecewise-constant signal at RMSE **0.0178** against Fourier's **0.1387**, nearly **8× better**. On the plot, the Fourier reconstruction rings visibly around every step while the Haar version keeps the edges sharp.

**Why Fourier struggles is a theorem, not bad luck.** A step discontinuity is not sparse in a sinusoidal basis — building a sharp edge from smooth sinusoids takes very many of them, and truncating the series produces the overshoot and ringing known as **Gibbs' phenomenon**, which the room met in [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb). Forty coefficients simply cannot represent five clean edges.

Haar's basis functions *are* steps. A piecewise-constant signal is therefore sparse in it: a handful of large coefficients capture the plateaus and the transitions, and everything else is near zero. Same signal, same budget, and the difference is entirely which basis the signal happens to be sparse in.

**Now the converse, so nobody over-generalises.** Run this on a pure sinusoid and the result inverts completely: Fourier needs *one* coefficient, while Haar needs many to approximate a smooth curve from blocky pieces. **Neither basis is better.** Each is sparse for a different class of signal, and the whole art is matching the basis to the content.

That is precisely the lesson [Sparse Dictionary Learning](./Sparse_Dictionary_Learning.ipynb) takes further — if no standard basis fits your data, *learn* one — and it is the same distinction as there between "can represent" and "can represent *briefly*". Both bases here are complete and could reproduce the signal exactly with all 512 coefficients. The question was never representability; it was sparsity.

**And this is why JPEG-2000 uses wavelets while JPEG uses the DCT.** Photographs contain edges, and edges are the thing wavelets encode cheaply — which is also why JPEG's block-DCT produces visible ringing around sharp boundaries at high compression, exactly the artifact in the orange trace above. The bake-off in this cell is, in miniature, the argument that changed image compression standards.

## 6. Conclusion

The z-transform's ROC settles stability vs causality; decimation and interpolation change rates honestly; polyphase makes it all nearly free; and wavelets re-spend the uncertainty budget where the signal needs it. This is the toolkit of every modern codec and SDR front-end.

---
## Where next

- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — multirate chains in the wild.
- [Compressed Sensing](./Compressed_Sensing.ipynb) — sparsity in a basis, weaponized.
- [Audio & Speech DSP](./Audio_Speech_DSP.ipynb) — the STFT/wavelet trade on real sound.